# Fraud detection — EDA

Goal of this notebook: understand the class imbalance, confirm where fraud actually occurs, sanity-check the balance columns, and validate that the prescribed time-ordered split is usable. Findings feed the feature work in `pipeline.py`.

Plots are written to `reports/`.

In [ ]:
import sys
from pathlib import Path

# Make src importable whether the notebook is launched from notebooks/ or the
# repo root.
_root = Path.cwd()
if _root.name == "notebooks":
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_data, repo_root

sns.set_theme(style="whitegrid")
REPORTS = repo_root() / "reports"
REPORTS.mkdir(exist_ok=True)

df = load_data()  # full ~6.36M rows
print(df.shape)
df.head()

## 1. Class balance

Accuracy is meaningless here, so we establish the positive rate up front.

In [ ]:
counts = df["isFraud"].value_counts()
print(counts)
print(f"fraud rate: {df['isFraud'].mean():.4%}")

## 2. Fraud rate by transaction type

Domain claim to verify: fraud only happens in `TRANSFER` and `CASH_OUT`.

In [ ]:
by_type = df.groupby("type", observed=True)["isFraud"].agg(["count", "sum", "mean"])
by_type = by_type.rename(columns={"count": "n", "sum": "frauds", "mean": "fraud_rate"})
print(by_type.sort_values("fraud_rate", ascending=False))

fig, ax = plt.subplots(figsize=(7, 4))
(by_type["fraud_rate"] * 100).sort_values().plot.barh(ax=ax, color="#c0392b")
ax.set_xlabel("fraud rate (%)")
ax.set_title("Fraud rate by transaction type")
fig.tight_layout()
fig.savefig(REPORTS / "fraud_rate_by_type.png", dpi=120)
plt.show()

In [ ]:
# Confirm: every fraud row is TRANSFER or CASH_OUT.
print(df.loc[df.isFraud == 1, "type"].value_counts())

## 3. isFlaggedFraud

The business rule (transfer > 200k). Checking how often it fires and whether it is useful.

In [ ]:
print("flagged total:", int(df.isFlaggedFraud.sum()))
print(pd.crosstab(df.isFlaggedFraud, df.isFraud))

## 4. Amount distribution

Heavily right-skewed, so plotted on a log scale. Fraud amounts sit much higher.

In [ ]:
print(df.groupby("isFraud", observed=True)["amount"].describe())

fig, ax = plt.subplots(figsize=(7, 4))
for label, color in [(0, "#2980b9"), (1, "#c0392b")]:
    vals = df.loc[df.isFraud == label, "amount"]
    vals = vals[vals > 0]
    ax.hist(np.log10(vals), bins=60, alpha=0.5, density=True,
            label=("legit" if label == 0 else "fraud"), color=color)
ax.set_xlabel("log10(amount)")
ax.set_ylabel("density")
ax.set_title("Transaction amount: fraud vs legit")
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / "amount_distribution.png", dpi=120)
plt.show()

## 5. Balance-error sanity check

The raw balance columns are a leakage trap. The real signal is whether the
transaction *reconciles*:

- `errorBalanceOrig = newbalanceOrig + amount - oldbalanceOrg`
- `errorBalanceDest = oldbalanceDest + amount - newbalanceDest`

For a clean transaction these are ~0. We check how often they aren't, and
whether the error separates fraud from legit (restricted to the only two types
that carry fraud).

In [ ]:
df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

tol = 0.01
print("orig non-reconcile rate (all):", (df.errorBalanceOrig.abs() > tol).mean())
print("dest non-reconcile rate (all):", (df.errorBalanceDest.abs() > tol).mean())

sub = df[df.type.isin(["TRANSFER", "CASH_OUT"])]
print("\nmean errorBalanceOrig by class (TRANSFER/CASH_OUT):")
print(sub.groupby("isFraud", observed=True)["errorBalanceOrig"].mean())
print("\nmean errorBalanceDest by class (TRANSFER/CASH_OUT):")
print(sub.groupby("isFraud", observed=True)["errorBalanceDest"].mean())

In [ ]:
# Clip to a readable range for the plot; the tails are extreme.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["errorBalanceOrig", "errorBalanceDest"]):
    data = [sub.loc[sub.isFraud == 0, col].clip(-2e6, 2e6),
            sub.loc[sub.isFraud == 1, col].clip(-2e6, 2e6)]
    ax.boxplot(data, labels=["legit", "fraud"], showfliers=False)
    ax.set_title(col)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
fig.suptitle("Balance error by class (TRANSFER / CASH_OUT, clipped)")
fig.tight_layout()
fig.savefig(REPORTS / "balance_error.png", dpi=120)
plt.show()

## 6. Zero-balance and merchant patterns

Two cheap flags worth engineering later.

In [ ]:
print("dest oldbalance==0 rate by class (T/CO):")
print(sub.assign(z=sub.oldbalanceDest == 0).groupby("isFraud", observed=True)["z"].mean())

df["isMerchantDest"] = df["nameDest"].str.startswith("M")
print("\nfraud count among merchant destinations:", int(df.loc[df.isMerchantDest, "isFraud"].sum()))

## 7. Fraud over time and the prescribed split

The brief fixes a time-ordered split (4M train / 1M eval / ~1.36M prod). Checking
how fraud is distributed across it — this matters because fraud density is not
stationary.

In [ ]:
for name, lo, hi in [("train", 0, 4_000_000), ("eval", 4_000_000, 5_000_000), ("prod", 5_000_000, len(df))]:
    part = df.iloc[lo:hi]
    print(f"{name:6s} rows={len(part):>8} frauds={int(part.isFraud.sum()):>5} rate={part.isFraud.mean():.4%}")

per_step = df.groupby("step", observed=True)["isFraud"].sum()
fig, ax = plt.subplots(figsize=(9, 4))
per_step.plot(ax=ax, color="#c0392b", lw=0.8)
# split boundaries by step (approx, since rows are step-ordered)
ax.axvline(df.iloc[4_000_000]["step"], color="k", ls="--", lw=0.8, label="train|eval")
ax.axvline(df.iloc[5_000_000]["step"], color="grey", ls="--", lw=0.8, label="eval|prod")
ax.set_xlabel("step (hour)")
ax.set_ylabel("fraud count")
ax.set_title("Fraud count per step, with split boundaries")
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / "fraud_over_time.png", dpi=120)
plt.show()

## Findings

See `reports/EDA_FINDINGS.md` for the written summary.